# GANForge -- CycleGAN (Standalone Notebook)

**Unpaired image-to-image translation**

This is a **self-contained extract** of the CycleGAN section from
the [GANForge Master Notebook](GANForge_Master_Notebook.ipynb) -- every
shared setup utility it needs (imports, device selection, seeding,
`weights_init`, `show_images`, checkpointing, etc.) is included below in
Section 0, so this notebook runs **top to bottom on its own**, in Colab
or locally, without needing any other file.

> **⚠️ Compute note.** See the Section 0 GPU warning below. Heavier
> architectures (Progressive GAN, StyleGAN2, BigGAN, StackGAN, GameGAN)
> include a lightweight from-scratch training loop sized for a single
> GPU (or CPU, slowly) plus, where realistically available, code to load
> strong pretrained weights for inference.

<a id="0"></a>
## 0. Global Setup & Utilities

Everything shared across all twelve GANs lives here: installs, imports,
device selection, seeding, and small helper functions (`weights_init`,
`show_images`, `save_checkpoint`, `Logger`, an `EMA` helper, and a spectral
-norm-aware conv wrapper). Re-using these keeps every later section short
and focused on what makes *that* GAN different.

**🚀 Pro Tip.** In Google Colab: `Runtime -> Change runtime type -> T4 GPU`
before running anything. Everything here auto-detects CUDA and degrades
gracefully to CPU (slow but functional) if none is found.

In [ ]:
# Colab-only: uncomment to install packages not preinstalled in Colab.
# All of these are optional -- every dataset loader below has an offline
# synthetic fallback, so the notebook still runs without them.
# !pip install -q torchaudio datasets soundfile pillow-simd torchmetrics
# !pip install -q pytorch-fid clean-fid


In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------
import os
import io
import math
import time
import random
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T
import torchvision.utils as vutils

import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except ImportError:
    # Minimal shim so the notebook still runs if tqdm is somehow missing.
    def tqdm(iterable, *args, **kwargs):
        return iterable

warnings.filterwarnings("ignore")  # GAN training is noisy (deprecation spam); keep output readable

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)


In [ ]:
# ---------------------------------------------------------------------------
# Device setup
# ---------------------------------------------------------------------------
# We centralize device selection so every model/tensor in the notebook is
# created consistently. All training loops below check DEVICE.type == "cuda"
# to decide whether to use mixed precision (torch.cuda.amp), which roughly
# halves memory use and often speeds up training 2-3x on modern GPUs.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

if DEVICE.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found -- running on CPU. Reduce EPOCHS/BATCH_SIZE in "
          "every section for reasonable runtimes.")


In [ ]:
# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
# GANs are notoriously sensitive to initialization -- fixing every RNG makes
# results reproducible across reruns, which matters a lot when you are
# debugging "why did my generator collapse this time".
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [ ]:
# ---------------------------------------------------------------------------
# Shared helper functions used across every GAN section below
# ---------------------------------------------------------------------------

def weights_init(m: nn.Module):
    '''DCGAN-style weight init (Radford et al., 2015).

    Why: GAN training is a saddle-point optimization between two networks
    that constantly move each other's loss landscape. Starting both nets
    from small, zero-centered weights (N(0, 0.02)) avoids one network
    dominating early purely due to init scale, which is a common cause of
    early, unrecoverable mode collapse.

    Uses isinstance (NOT a substring match on the class name) because many
    custom container classes in this notebook -- PGAN_ConvBlock,
    ModulatedConv2d, StyleConvBlock -- have "Conv" in their own name; a
    naive `"Conv" in classname` check would incorrectly try to initialize
    THOSE containers directly (they have no `.weight` of their own) instead
    of only their actual nn.Conv2d/nn.Linear/nn.BatchNorm2d leaf children.
    '''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None:
            nn.init.constant_(m.bias.data, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)) and getattr(m, "affine", False):
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def show_images(images: torch.Tensor, title: str = "", nrow: int = 8, figsize=(8, 8), normalize=True):
    '''Display a batch of image tensors (B, C, H, W) in [-1, 1] or [0, 1] as a grid.'''
    images = images.detach().cpu()
    grid = vutils.make_grid(images, nrow=nrow, normalize=normalize, value_range=(-1, 1) if normalize else None, padding=2)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.title(title)
    npimg = grid.numpy()
    if npimg.shape[0] == 1:
        plt.imshow(npimg[0], cmap="gray")
    else:
        plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()


def save_checkpoint(state: dict, path: str):
    '''Persist model/optimizer state so long training runs survive a Colab disconnect.'''
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)


def load_checkpoint(path: str, map_location=None):
    if not os.path.exists(path):
        return None
    return torch.load(path, map_location=map_location or DEVICE)


class EMA:
    '''Exponential Moving Average of generator weights.

    Why: instantaneous generator weights oscillate a lot during adversarial
    training (they are chasing a moving discriminator). Averaging weights
    over time (as StyleGAN2 and BigGAN both do) gives a much smoother,
    higher-quality generator for *inference*, at zero extra training cost.
    '''
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone()

    def copy_to(self, model: nn.Module):
        model.load_state_dict(self.shadow, strict=True)


class AvgMeter:
    '''Running mean tracker for loss logging.'''
    def __init__(self):
        self.sum, self.n = 0.0, 0

    def update(self, val, k=1):
        self.sum += float(val) * k
        self.n += k

    @property
    def avg(self):
        return self.sum / max(self.n, 1)


def denorm(x: torch.Tensor) -> torch.Tensor:
    '''Map a tensor from [-1, 1] (tanh output range) back to [0, 1] for display/saving.'''
    return (x.clamp(-1, 1) + 1) / 2


CHECKPOINT_DIR = "checkpoints"
SAMPLES_DIR = "samples"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)
print("Shared utilities ready.")


<a id="8"></a>
## 8. CycleGAN -- Unpaired image-to-image translation

### 8.1 Problem Statement & Why this GAN

Zhu et al. (2017) addressed image-to-image translation (horses <-> zebras,
photos <-> paintings) in the setting where you have **two unpaired
collections** of images (a bunch of horse photos, a bunch of zebra photos,
*no* horse/zebra pairs of the exact same pose) -- pix2pix-style
paired translation is inapplicable, since no ground-truth target exists
for any given input. CycleGAN's solution is **cycle-consistency**: learn
two generators `G: A->B` and `F: B->A` such that translating there and
back recovers the original image, `F(G(a)) ~ a` and `G(F(b)) ~ b`. This
constraint alone, with no paired supervision at all, is enough to force
`G`/`F` to preserve content while changing only style/domain-specific
appearance. This is the **unpaired image-to-image translation** problem.

### 8.2 Data Collection & Preparation

In [ ]:
CYCLEGAN_IMG_SIZE = 64
CYCLEGAN_BATCH_SIZE = 8
CYCLEGAN_EPOCHS = 6

cyclegan_transform = T.Compose([T.Resize(CYCLEGAN_IMG_SIZE), T.CenterCrop(CYCLEGAN_IMG_SIZE),
                                 T.ToTensor(), T.Normalize((0.5,) * 3, (0.5,) * 3)])

class UnpairedFolderDataset(Dataset):
    '''Loads all images directly under `root` (any format Pillow supports);
    CRUCIALLY, this dataset is indexed independently for domain A and
    domain B -- we never assume image i in A corresponds to image i in B.'''
    def __init__(self, root, transform):
        from PIL import Image
        self.Image = Image
        self.paths = sorted(Path(root).glob("*.jpg")) + sorted(Path(root).glob("*.png"))
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = self.Image.open(self.paths[i]).convert("RGB")
        return self.transform(img)

def add_stripes(img: torch.Tensor, rng) -> torch.Tensor:
    '''Procedurally overlay a zebra-like diagonal stripe pattern -- used
    ONLY to build an offline, guaranteed-available "second domain" when the
    real horse2zebra dataset cannot be downloaded, so cycle-consistency
    still has a genuine (if synthetic) style-transfer task to learn.'''
    c, h, w = img.shape
    yy, xx = torch.meshgrid(torch.arange(h), torch.arange(w), indexing="ij")
    phase = rng.uniform(0, 2 * math.pi)
    freq = rng.uniform(0.4, 0.7)
    stripes = (torch.sin(freq * (xx + yy).float() + phase) > 0).float()
    stripe_color = torch.tensor([-1.0, -1.0, -1.0]).view(3, 1, 1)
    return img * (1 - 0.5 * stripes) + stripe_color * (0.5 * stripes)

try:
    import urllib.request, zipfile
    zip_path = "data/horse2zebra.zip"
    if not os.path.exists("data/horse2zebra"):
        os.makedirs("data", exist_ok=True)
        url = "https://efrosgans.eecs.berkeley.edu/cyclegan/datasets/horse2zebra.zip"
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall("data")
    domain_A = UnpairedFolderDataset("data/horse2zebra/trainA", cyclegan_transform)
    domain_B = UnpairedFolderDataset("data/horse2zebra/trainB", cyclegan_transform)
    assert len(domain_A) > 0 and len(domain_B) > 0
    print(f"Loaded real horse2zebra: {len(domain_A)} horses, {len(domain_B)} zebras")
except Exception as e:
    print(f"horse2zebra download failed ({e}); falling back to synthetic "
          f"'plain photo' <-> 'striped photo' domains built from CIFAR-10.")
    try:
        cifar_src = torchvision.datasets.CIFAR10(root="data", train=True, download=True,
                                                   transform=T.Compose([T.Resize(CYCLEGAN_IMG_SIZE), T.ToTensor(),
                                                                         T.Normalize((0.5,) * 3, (0.5,) * 3)]))
        horse_idx = [i for i, (_, y) in enumerate(cifar_src) if y == 7]  # CIFAR-10 class 7 = horse
    except Exception:
        cifar_src = None
        horse_idx = []
    rng = np.random.default_rng(SEED)
    if cifar_src is not None and len(horse_idx) >= 200:
        half = len(horse_idx) // 2
        imgs_A = torch.stack([cifar_src[i][0] for i in horse_idx[:half]])
        imgs_B = torch.stack([add_stripes(cifar_src[i][0], rng) for i in horse_idx[half:]])
    else:
        imgs_A = torch.rand(200, 3, CYCLEGAN_IMG_SIZE, CYCLEGAN_IMG_SIZE) * 2 - 1
        imgs_B = torch.stack([add_stripes(img, rng) for img in imgs_A])
    domain_A = TensorDataset(imgs_A)
    domain_B = TensorDataset(imgs_B)

def _unwrap(batch):
    return batch[0] if isinstance(batch, (list, tuple)) else batch

loader_A = DataLoader(domain_A, batch_size=CYCLEGAN_BATCH_SIZE, shuffle=True, drop_last=True)
loader_B = DataLoader(domain_B, batch_size=CYCLEGAN_BATCH_SIZE, shuffle=True, drop_last=True)

show_images(torch.stack([_unwrap(domain_A[i]) for i in range(8)]), title="Domain A samples", nrow=4)
show_images(torch.stack([_unwrap(domain_B[i]) for i in range(8)]), title="Domain B samples", nrow=4)


### 8.3 Model Architecture

In [ ]:
class CycleGAN_ResBlock(nn.Module):
    '''Reflection-padded residual block -- CycleGAN's generator backbone.
    Reflection padding (rather than zero padding) avoids the border
    artifacts a translation network would otherwise learn near image
    edges, since edges get "seen" less often during training.'''
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(ch, ch, 3), nn.InstanceNorm2d(ch), nn.ReLU(True),
            nn.ReflectionPad2d(1), nn.Conv2d(ch, ch, 3), nn.InstanceNorm2d(ch),
        )

    def forward(self, x):
        return x + self.block(x)


class CycleGAN_Generator(nn.Module):
    '''Encoder (downsample x2) -> N residual blocks -> Decoder (upsample x2).
    InstanceNorm (not BatchNorm) throughout is the CycleGAN/pix2pix
    standard: normalizing PER IMAGE rather than per batch matters for
    style-transfer tasks, where each image's own contrast/brightness
    should be treated independently, not tied to other images in the batch.'''
    def __init__(self, nc=3, ngf=64, n_res_blocks=6):
        super().__init__()
        layers = [nn.ReflectionPad2d(3), nn.Conv2d(nc, ngf, 7), nn.InstanceNorm2d(ngf), nn.ReLU(True)]
        ch = ngf
        for _ in range(2):   # downsample 64 -> 32 -> 16
            layers += [nn.Conv2d(ch, ch * 2, 3, 2, 1), nn.InstanceNorm2d(ch * 2), nn.ReLU(True)]
            ch *= 2
        for _ in range(n_res_blocks):
            layers += [CycleGAN_ResBlock(ch)]
        for _ in range(2):   # upsample back 16 -> 32 -> 64
            layers += [nn.ConvTranspose2d(ch, ch // 2, 3, 2, 1, output_padding=1), nn.InstanceNorm2d(ch // 2), nn.ReLU(True)]
            ch //= 2
        layers += [nn.ReflectionPad2d(3), nn.Conv2d(ch, nc, 7), nn.Tanh()]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


class CycleGAN_PatchDiscriminator(nn.Module):
    '''70x70 PatchGAN: instead of one real/fake logit for the whole image,
    output a GRID of logits, each judging a local patch. This focuses the
    discriminator (and hence the generator's gradient) on local texture
    realism rather than global layout -- exactly the right inductive bias
    for style/texture translation tasks.'''
    def __init__(self, nc=3, ndf=64):
        super().__init__()
        def block(in_c, out_c, stride=2, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, stride, 1)]
            if norm:
                layers.append(nn.InstanceNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, True))
            return layers
        self.net = nn.Sequential(
            *block(nc, ndf, norm=False),
            *block(ndf, ndf * 2),
            *block(ndf * 2, ndf * 4),
            *block(ndf * 4, ndf * 8, stride=1),
            nn.Conv2d(ndf * 8, 1, 4, 1, 1),
        )

    def forward(self, x):
        return self.net(x)   # (B, 1, H', W') patch map of logits


G_AB = CycleGAN_Generator().to(DEVICE).apply(weights_init)   # horse -> zebra
G_BA = CycleGAN_Generator().to(DEVICE).apply(weights_init)   # zebra -> horse
D_A = CycleGAN_PatchDiscriminator().to(DEVICE).apply(weights_init)
D_B = CycleGAN_PatchDiscriminator().to(DEVICE).apply(weights_init)


### 8.4 Loss Functions & Optimizers

In [ ]:
# LSGAN (least-squares) adversarial loss: CycleGAN's actual choice --
# MSE against 1/0 targets instead of BCE. Empirically gives more stable
# gradients for image-translation tasks than the vanilla log-loss.
cyclegan_adv_criterion = nn.MSELoss()
cyclegan_cycle_criterion = nn.L1Loss()
cyclegan_identity_criterion = nn.L1Loss()

CYCLEGAN_LAMBDA_CYCLE = 10.0
CYCLEGAN_LAMBDA_IDENTITY = 5.0   # weight on identity loss relative to cycle loss (paper: 0.5x lambda_cycle)

cyclegan_opt_G = torch.optim.Adam(list(G_AB.parameters()) + list(G_BA.parameters()), lr=2e-4, betas=(0.5, 0.999))
cyclegan_opt_D = torch.optim.Adam(list(D_A.parameters()) + list(D_B.parameters()), lr=2e-4, betas=(0.5, 0.999))

class ImagePool:
    '''History buffer of previously generated fakes (Shrivastava et al.,
    2017, adopted by CycleGAN): update discriminators using a MIX of the
    latest fakes and older ones sampled from this pool, rather than only
    the latest batch. This damps the oscillation that can occur when D is
    trained exclusively against G's *current* (fast-moving) outputs.'''
    def __init__(self, pool_size=50):
        self.pool_size = pool_size
        self.images = []

    def query(self, images):
        if self.pool_size == 0:
            return images
        result = []
        for img in images:
            img = img.unsqueeze(0)
            if len(self.images) < self.pool_size:
                self.images.append(img)
                result.append(img)
            elif random.random() > 0.5:
                idx = random.randrange(self.pool_size)
                result.append(self.images[idx].clone())
                self.images[idx] = img
            else:
                result.append(img)
        return torch.cat(result, dim=0)

pool_A, pool_B = ImagePool(), ImagePool()


### 8.5 Training Loop

In [ ]:
cyclegan_history = []
for epoch in range(CYCLEGAN_EPOCHS):
    meters = {"G": AvgMeter(), "D": AvgMeter(), "cycle": AvgMeter()}
    pbar = tqdm(zip(loader_A, loader_B), total=min(len(loader_A), len(loader_B)),
                desc=f"[CycleGAN] epoch {epoch+1}/{CYCLEGAN_EPOCHS}")
    for real_A, real_B in pbar:
        real_A, real_B = _unwrap(real_A).to(DEVICE), _unwrap(real_B).to(DEVICE)
        b = real_A.size(0)

        # ---- Generators: adversarial + cycle-consistency + identity ----
        cyclegan_opt_G.zero_grad(set_to_none=True)
        fake_B = G_AB(real_A)
        fake_A = G_BA(real_B)
        loss_adv_AB = cyclegan_adv_criterion(D_B(fake_B), torch.ones_like(D_B(fake_B)))
        loss_adv_BA = cyclegan_adv_criterion(D_A(fake_A), torch.ones_like(D_A(fake_A)))

        recon_A = G_BA(fake_B)   # A -> B -> A
        recon_B = G_AB(fake_A)   # B -> A -> B
        loss_cycle = cyclegan_cycle_criterion(recon_A, real_A) + cyclegan_cycle_criterion(recon_B, real_B)

        # Identity loss: G_AB(real_B) should be ~real_B (already a zebra,
        # zebra->zebra should be near-identity) -- empirically helps
        # preserve color composition, especially for photo<->painting tasks.
        loss_idt = cyclegan_identity_criterion(G_AB(real_B), real_B) + cyclegan_identity_criterion(G_BA(real_A), real_A)

        loss_g = loss_adv_AB + loss_adv_BA + CYCLEGAN_LAMBDA_CYCLE * loss_cycle + CYCLEGAN_LAMBDA_IDENTITY * loss_idt
        loss_g.backward()
        cyclegan_opt_G.step()

        # ---- Discriminators (using the image pool) ----
        cyclegan_opt_D.zero_grad(set_to_none=True)
        fake_B_pooled = pool_B.query(fake_B.detach())
        fake_A_pooled = pool_A.query(fake_A.detach())
        loss_d_A = 0.5 * (cyclegan_adv_criterion(D_A(real_A), torch.ones_like(D_A(real_A))) +
                           cyclegan_adv_criterion(D_A(fake_A_pooled), torch.zeros_like(D_A(fake_A_pooled))))
        loss_d_B = 0.5 * (cyclegan_adv_criterion(D_B(real_B), torch.ones_like(D_B(real_B))) +
                           cyclegan_adv_criterion(D_B(fake_B_pooled), torch.zeros_like(D_B(fake_B_pooled))))
        loss_d = loss_d_A + loss_d_B
        loss_d.backward()
        cyclegan_opt_D.step()

        meters["G"].update(loss_g.item()); meters["D"].update(loss_d.item()); meters["cycle"].update(loss_cycle.item())
        pbar.set_postfix(loss_G=f"{loss_g.item():.3f}", loss_D=f"{loss_d.item():.3f}", cycle=f"{loss_cycle.item():.3f}")

    cyclegan_history.append((meters["G"].avg, meters["D"].avg, meters["cycle"].avg))
    with torch.no_grad():
        G_AB.eval()
        demo_fake_B = G_AB(real_A[:8])
        G_AB.train()
    show_images(torch.cat([real_A[:8], demo_fake_B]), title=f"CycleGAN A->B after epoch {epoch+1} (top=real A, bottom=fake B)", nrow=8)
    save_checkpoint({"G_AB": G_AB.state_dict(), "G_BA": G_BA.state_dict(), "D_A": D_A.state_dict(), "D_B": D_B.state_dict()},
                     f"{CHECKPOINT_DIR}/cyclegan_epoch{epoch+1}.pt")

print("CycleGAN training complete.")


### 8.6 Inference / Generation Function

In [ ]:
@torch.no_grad()
def cyclegan_translate(images: torch.Tensor, direction: str = "A2B") -> torch.Tensor:
    '''Clean translation entry point. direction: "A2B" or "B2A".'''
    images = images.to(DEVICE)
    if direction == "A2B":
        G_AB.eval(); out = G_AB(images); G_AB.train()
    else:
        G_BA.eval(); out = G_BA(images); G_BA.train()
    return out

sample_A = torch.stack([_unwrap(domain_A[i]) for i in range(8)])
show_images(torch.cat([sample_A, cyclegan_translate(sample_A, "A2B")]),
            title="CycleGAN inference: top=input (A), bottom=translated (B)", nrow=8)


### 8.7 Visualization & Results

In [ ]:
gs, ds, cs = zip(*cyclegan_history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(gs, label="G"); axes[0].plot(ds, label="D"); axes[0].legend(); axes[0].set_title("Adversarial losses")
axes[1].plot(cs); axes[1].set_title("Cycle-consistency L1 loss (should trend down)")
plt.show()

# Full round trip: A -> B -> A, to visually verify cycle-consistency held.
sample_A8 = torch.stack([_unwrap(domain_A[i]) for i in range(4)])
fake_B = cyclegan_translate(sample_A8, "A2B")
recon_A = cyclegan_translate(fake_B, "B2A")
show_images(torch.cat([sample_A8, fake_B, recon_A]),
            title="Round trip: real A (top) -> fake B (mid) -> reconstructed A (bottom)", nrow=4, figsize=(6, 5))


### 8.8 User Input → Output Demo

In [ ]:
IMAGE_INDEX = 3          # which image from domain A to translate
DIRECTION = "A2B"        # "A2B" or "B2A"

user_img = _unwrap(domain_A[IMAGE_INDEX]).unsqueeze(0)
translated = cyclegan_translate(user_img, DIRECTION)
show_images(torch.cat([user_img, translated]), title=f"User demo: direction={DIRECTION}", nrow=2, figsize=(4, 2))


### 8.9 Key Insights & Limitations of this GAN

- 💡 **Insight.** Cycle-consistency is a form of **self-supervision**: it
  manufactures a training signal (the reconstruction target *is* the
  input) in a setting where no direct supervision (paired data) exists at
  all -- a pattern reused far beyond GANs (e.g. in unsupervised
  representation learning).
- ⚠️ **Warning.** Cycle-consistency only constrains *content preservation*,
  not *correctness of the translation* -- CycleGAN can and does learn
  degenerate solutions that hide information imperceptibly in the "fake"
  image purely to make the cycle reconstruct well (documented as steganographic
  artifacts in follow-up work), rather than truly discarding
  domain-A-specific content.
- ⚠️ **Warning.** Because there is no pairing constraint, CycleGAN cannot
  guarantee *geometric* consistency (e.g. it may translate texture well
  but distort shape) -- this is a known limitation the PatchGAN's local
  focus can actually make worse, since it never "sees" global structure.

**This GAN solved the problem of: translating images between two visual
domains using only unpaired example collections from each domain, with no
example ever telling the model "this A corresponds to this B".**